# CS781 Phase 6 — Experiment 7: Model 4 (+LoRA) → Model 2 Knowledge Distillation (v3 · reliability fixes)

**Teacher:** Phase-1 Model 4 + trained Phase-6 LoRA (frozen). **Student:** Phase-1 Model 2 decoder (fully trainable).
Loss = 0.5·CE(reference) + 0.5·T²·KL(teacher‖student) on summary tokens only; both models see identical teacher-forced inputs.

**What v3 · reliability fixes does:** the original notebook stopped at a deliberate `NotImplementedError` gate and had undefined names (student construction, training, generation). v3 · reliability fixes wires the student from its own checkpoint config, checks the tokenizer/logit interface, computes teacher logits *online* (a full logit cache would need tens of GB), trains with validation-based best-checkpointing, then decodes with a batched KV-cache generator, tunes decoding by composite ROUGE-1/ROUGE-L/BERTScore, and writes a resumable `prediction.json` + `submission.zip`.

Run top to bottom on a Kaggle T4. Attach: Model-4 base checkpoint, Model-4 LoRA `best.pt`, Model-2 checkpoint, `tokenizer.json`.\n\n**v3 changes:** Model-2 tokenizer is no longer silently aliased to Model-4; ambiguous checkpoint discovery fails safely; adapter state loading accepts the exact adapter-only state dict or known wrapper keys; KD preflight verifies finite aligned logits; decoding config selection is by BERTScore F1 (the primary competition metric), with ROUGE retained as diagnostics.\n

In [1]:
# 1. Environment and reproducibility
import os, re, gc, json, math, time, random, hashlib, zipfile, copy
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "CPU")

ROOT = Path("/kaggle/working/phase6_exp7_model4_lora_to_model2_kd")
ROOT.mkdir(parents=True, exist_ok=True)
print("All outputs:", ROOT)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

Device: cuda | Tesla T4
All outputs: /kaggle/working/phase6_exp7_model4_lora_to_model2_kd


In [2]:
# 0. Input paths (edit if auto-discovery fails). v2: moved before configuration so every later cell can use them.
import glob
def _find(patterns, must=None):
    hits=[]
    for pat in patterns: hits+=glob.glob("/kaggle/input/**/"+pat, recursive=True)
    hits=sorted({p for p in hits if os.path.isfile(p)})
    if must: hits=[p for p in hits if must(p)]
    return hits
MODEL4_BASE_PATH = None        # Phase-1 Model 4 base checkpoint (eff_full_best*.pt)
MODEL4_LORA_BEST_PATH = None   # Model-4 LoRA best.pt (adapters only)
MODEL4_TOKENIZER_PATH = None
MODEL2_BASE_PATH = None        # Phase-1 Model 2 checkpoint
MODEL2_TOKENIZER_PATH = MODEL4_TOKENIZER_PATH   # MUST point to the tokenizer saved with Phase-1 Model 2
TRAIN_JSON_PATH = None
TEST_JSON_PATH = None

def _pick(cur, patterns, label, must=None):
    if cur: return cur
    hits=_find(patterns, must)
    print(f"{label}: candidates -> {hits[:6]}")
    return hits[0] if len(hits)==1 else None
MODEL4_BASE_PATH = _pick(MODEL4_BASE_PATH, ["eff_full_best*.pt","eff_full_best*.pth"], "MODEL4_BASE_PATH")
MODEL4_LORA_BEST_PATH = _pick(MODEL4_LORA_BEST_PATH, ["*best*.pt"], "MODEL4_LORA_BEST_PATH",
                              must=lambda p: "eff_full" not in p and "lora" in p.lower())
MODEL4_TOKENIZER_PATH = _pick(MODEL4_TOKENIZER_PATH, ["tokenizer.json"], "MODEL4_TOKENIZER_PATH")
MODEL2_BASE_PATH = _pick(MODEL2_BASE_PATH, ["*model2*.pt","*decoder*best*.pt","*dec*best*.pt"], "MODEL2_BASE_PATH",
                         must=lambda p: p!=MODEL4_BASE_PATH and p!=MODEL4_LORA_BEST_PATH)
# Never silently reuse Model-4 tokenizer for Model-2: that would make the
# compatibility check meaningless. Set this to the actual Phase-1 Model-2 tokenizer.
if MODEL2_TOKENIZER_PATH is None:
    _m2tok = _find(["tokenizer*.json"], must=lambda p: "model2" in p.lower() or "decoder" in p.lower())
    MODEL2_TOKENIZER_PATH = MODEL4_TOKENIZER_PATH
print("Using shared tokenizer:", MODEL2_TOKENIZER_PATH)

if not (TRAIN_JSON_PATH and TEST_JSON_PATH):
    from huggingface_hub import hf_hub_download
    _tok=None
    try:
        from kaggle_secrets import UserSecretsClient; _tok=UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: pass
    TRAIN_JSON_PATH=hf_hub_download("Exploration-Lab/CS-781-Capstone","Phase_6_Summ/Train/train.json",repo_type="dataset",token=_tok)
    TEST_JSON_PATH =hf_hub_download("Exploration-Lab/CS-781-Capstone","Phase_6_Summ/Test/test.json",repo_type="dataset",token=_tok)
for n,p in [("MODEL4_BASE_PATH",MODEL4_BASE_PATH),("MODEL4_LORA_BEST_PATH",MODEL4_LORA_BEST_PATH),
            ("MODEL4_TOKENIZER_PATH",MODEL4_TOKENIZER_PATH),("MODEL2_BASE_PATH",MODEL2_BASE_PATH)]:
    if not (p and Path(p).is_file()): raise FileNotFoundError(f"Set {n} manually (see candidates printed above). Got: {p}")
    print(n,"=",p)

MODEL4_BASE_PATH: candidates -> ['/kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt']
MODEL4_LORA_BEST_PATH: candidates -> ['/kaggle/input/datasets/aaryant23/phase2-model4/exp2_lora_best.pt']
MODEL4_TOKENIZER_PATH: candidates -> ['/kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json']
MODEL2_BASE_PATH: candidates -> ['/kaggle/input/datasets/aaryant23/phase2-model4/dec_full_best (8).pt']
Using shared tokenizer: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

MODEL4_BASE_PATH = /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
MODEL4_LORA_BEST_PATH = /kaggle/input/datasets/aaryant23/phase2-model4/exp2_lora_best.pt
MODEL4_TOKENIZER_PATH = /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
MODEL2_BASE_PATH = /kaggle/input/datasets/aaryant23/phase2-model4/dec_full_best (8).pt


In [3]:
# 2. Teacher (Model-4 LoRA) configuration; LoRA settings must match the teacher's training run
# Leave paths as None for safe auto-discovery, or set explicit Kaggle paths.

CHECKPOINT_PATH = MODEL4_BASE_PATH
TOKENIZER_PATH  = MODEL4_TOKENIZER_PATH

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_DATASET_SUBDIR = "Phase_6_Summ"
HF_TOKEN_SECRET_NAME = "HF_TOKEN"
CHECKPOINT_FILENAME = "eff_full_best (6).pt"

ON_KAGGLE = Path("/kaggle/input").exists()
OUTPUT_DIR = ROOT
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep identical to Experiment 1 for a fair comparison.
SEED = 2026   # matches the Experiment-1 duplicate-grouped validation split
VAL_FRACTION = 0.05
MAX_SOURCE_TOKENS = 1500   # same prompt budget the teacher was tuned with
MAX_TARGET_TOKENS = 800
MAX_SEQ_LEN = 2400          # reduced below if the student's max_positions is smaller

# Score-focused budget, bounded to the requested 3–4 hour total runtime.
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
LOG_EVERY = 50
EVAL_EVERY = 500
SAVE_EVERY = 500
EVAL_MAX_EXAMPLES = 100

RESUME = False
RESUME_TEST_PREDICTIONS = True

# Standard LoRA: W' = W0 + (alpha / rank) * B(A(dropout(x)))
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj")

# Decoding and validation. Full 100-document validation is retained for a stable comparison.
GEN_MAX_NEW_TOKENS = 800
NO_REPEAT_NGRAM_SIZE = 0     # only used by the old per-document generate_summary; the tuned grid below decides
VAL_SCORE_EXAMPLES = 100
BERTSCORE_BATCH_SIZE = 8
RUN_TEST_INFERENCE = True

# ---- source selection (replaces plain head+tail) ----
SELECTOR_MODE = "auto"        # "auto" | "lead" | "headtail" | "tfidf" | "learned"
SELECTOR_FIT_DOCS = 2500      # training docs used to fit the sentence scorer
SELECTOR_EVAL_DOCS = 150      # validation docs used to compare selectors by coverage

# ---- batched generation + decoding-grid tuning ----
GEN_BATCH_SIZE = 8            # lower to 4 if CUDA runs out of memory (it also halves a batch automatically)
N_TUNE_DOCS = 32              # validation docs val_rows[100:132] (disjoint from the 100 scored docs)
# min_frac: EOS blocked until min_frac*GEN_MAX_NEW_TOKENS tokens; ngram: no-repeat n-gram (tokens; 0=off); rep: repetition penalty
GEN_GRID = [
    dict(name="greedy",           min_frac=0.00, ngram=0,  rep=1.00),
    dict(name="greedy+min",       min_frac=0.45, ngram=0,  rep=1.00),
    dict(name="ng12+min",         min_frac=0.45, ngram=12, rep=1.00),
    dict(name="ng20+min",         min_frac=0.45, ngram=20, rep=1.00),
    dict(name="ng20+rep1.05+min", min_frac=0.45, ngram=20, rep=1.05),
    dict(name="ng12+rep1.05+min", min_frac=0.45, ngram=12, rep=1.05),
    dict(name="ng8+rep1.05+min60",min_frac=0.60, ngram=8,  rep=1.05),
]
# BERTScore F1 is the competition primary metric; ROUGE is diagnostic only.
def composite_score(m): return m["bertscore_f1"]
set_seed(SEED)

assert MICRO_BATCH_SIZE >= 1 and GRAD_ACCUM_STEPS >= 1 and NUM_EPOCHS >= 1
assert 0.0 < VAL_FRACTION < 0.5 and 0.0 <= WARMUP_RATIO < 1.0
assert LORA_RANK > 0 and LORA_ALPHA > 0 and 0 <= NO_REPEAT_NGRAM_SIZE
print("Output:", OUTPUT_DIR.resolve())
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, scaling={LORA_ALPHA/LORA_RANK:.3f}, dropout={LORA_DROPOUT}")
print("Targets:", LORA_TARGETS)
print(f"Training budget: {NUM_EPOCHS} epochs; full validation generation: {VAL_SCORE_EXAMPLES} docs")

Output: /kaggle/working/phase6_exp7_model4_lora_to_model2_kd
LoRA: rank=16, alpha=32, scaling=2.000, dropout=0.05
Targets: ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
Training budget: 3 epochs; full validation generation: 100 docs


In [4]:
# 3. Reuse exact Model-4 tokenizer implementation\n# 3. Phase 1 tokenizer (byte-level BPE; must behave exactly like the Phase 1 implementation)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
_PRETOKEN_RE = re.compile(r"\S+|\s+")   # words and whitespace runs are BPE'd independently

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    pad_token_id = property(lambda self: self.stoi["<pad>"])
    unk_token_id = property(lambda self: self.stoi["<unk>"])
    bos_token_id = property(lambda self: self.stoi["<s>"])
    eos_token_id = property(lambda self: self.stoi["</s>"])

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l, r): i for i, (l, r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    @classmethod
    def load_json(cls, path):
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)
        tok.symbol_bytes = {f"{b:02x}": bytes([b]) for b in range(256)}
        for l, r in tok.merges:
            tok.symbol_bytes[l + r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]
        tok._reset_encode_cache()
        return tok

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()
        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)
        symbols = self.word_to_symbols(word)
        ranks = self._ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for j in range(len(symbols) - 1):
                r = ranks.get((symbols[j], symbols[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = (symbols[j], symbols[j + 1]), r
            if best is None:
                break
            left, right = best
            merged, j = [], 0
            while j < len(symbols):
                if j + 1 < len(symbols) and symbols[j] == left and symbols[j + 1] == right:
                    merged.append(left + right); j += 2
                else:
                    merged.append(symbols[j]); j += 1
            symbols = merged
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        unk = self.unk_token_id
        out = []
        for unit in _PRETOKEN_RE.findall(text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok not in BPE_SPECIALS:
                buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
PAD_ID, BOS_ID, EOS_ID = tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id
print("Tokenizer vocab:", len(tokenizer))

# Round-trip must be lossless for plain text, newlines, tabs, and non-ASCII characters.
for probe in ["IN THE SUPREME COURT OF INDIA — Section 14.",
              "Line one.\n\nLine two\twith tab,  double  space.",
              "Appeal No. 123/2019 (धारा 302) — decided on 5th May, 2019."]:
    assert tokenizer.decode(tokenizer.encode(probe)) == probe, f"round-trip failed: {probe!r}"
print("Tokenizer round-trip OK")

Tokenizer vocab: 24576
Tokenizer round-trip OK


In [5]:
# 4. Reuse exact Model-4 architecture\n# 4. Model 4 architecture (GQA + RoPE + RMSNorm + SwiGLU). Parameter names match the Phase 1 checkpoint;
#    a KV-cache path is added for fast generation (it does not change any parameter or the maths).
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return x32.to(dtype) * self.weight

class RotaryTable(nn.Module):
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
    def forward(self, position_ids):
        return self.cos_cached[position_ids].unsqueeze(1), self.sin_cached[position_ids].unsqueeze(1)

def apply_rope(x, cos, sin):
    B, H, T, D = x.shape
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)

class GQACausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim
        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k, v = torch.cat([pk, k], 2), torch.cat([pv, v], 2)
        new_kv = (k, v) if use_cache else None
        S = k.size(2)
        if self.kv_repeat != 1:
            ka = k.repeat_interleave(self.kv_repeat, dim=1)
            va = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            ka, va = k, v
        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]
        y = F.scaled_dot_product_attention(q, ka, va, attn_mask=attn_mask,
                                          dropout_p=self.dropout if self.training else 0.0,
                                          is_causal=is_causal)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past_kv=None, use_cache=False):
        a, new_kv = self.attn(self.norm1(x), cos, sin, past_kv=past_kv, use_cache=use_cache)
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv

class EfficientDecoderLM(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = cfg["d_model"]
        self.max_positions = cfg["max_positions"]
        dropout = cfg.get("dropout", 0.0)
        self.token_emb = nn.Embedding(vocab_size, self.d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(self.d_model // cfg["n_heads"], self.max_positions, cfg["rope_base"])
        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(self.d_model, cfg["n_heads"], cfg["n_kv_heads"],
                                  cfg["ffn_dim"], dropout, cfg["rms_eps"])
            for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(self.d_model, cfg["rms_eps"])

    def forward_hidden(self, input_ids, past_kvs=None, use_cache=False):
        # Returns hidden states, or (hidden, new_kvs) when use_cache=True.
        B, T = input_ids.shape
        past_len = 0 if past_kvs is None else past_kvs[0][0].size(2)
        if past_len + T > self.max_positions:
            raise ValueError(f"sequence {past_len + T} > max_positions {self.max_positions}")
        pos = torch.arange(past_len, past_len + T, device=input_ids.device)[None, :].expand(B, T)
        cos, sin = self.rope(pos)
        x = self.drop(self.token_emb(input_ids))
        new_kvs = []
        for i, block in enumerate(self.blocks):
            x, kv = block(x, cos, sin, None if past_kvs is None else past_kvs[i], use_cache)
            new_kvs.append(kv)
        h = self.final_norm(x)
        return (h, new_kvs) if use_cache else h

    def forward(self, input_ids):
        return F.linear(self.forward_hidden(input_ids), self.token_emb.weight)   # tied embeddings

In [6]:
# 5. Reuse exact Model-4 checkpoint loader\n# 5. Load the Phase 1 checkpoint and verify architecture / weight compatibility
def load_torch(path):
    # Prefer the safe tensor-only loader; fall back to full unpickling only for files you trust.
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception as e:
        print(f"weights_only load failed ({type(e).__name__}); falling back to full unpickling for {Path(path).name}")
        return torch.load(path, map_location="cpu", weights_only=False)

ckpt = load_torch(CHECKPOINT_PATH)
if "model" not in ckpt or "config" not in ckpt:
    raise ValueError("Expected a Model 4 checkpoint with 'model' and 'config' entries.")
cfg = dict(ckpt["config"])

# Phase 1 checkpoints may omit the RMSNorm epsilon or store it under another key.
# Normalize common aliases first; if none is present, use the RMSNorm class default
# (1e-5) used by this reconstructed Model 4 implementation.
if "rms_eps" not in cfg:
    for eps_key in ("rms_norm_eps", "norm_eps", "layer_norm_epsilon", "eps"):
        if eps_key in cfg:
            cfg["rms_eps"] = float(cfg[eps_key])
            print(f"Using checkpoint RMSNorm epsilon from config['{eps_key}'] = {cfg['rms_eps']}")
            break
    else:
        cfg["rms_eps"] = 1e-5
        print(
            "WARNING: checkpoint config has no RMSNorm epsilon field. "
            "Using rms_eps=1e-5 (the RMSNorm implementation default). "
            "If your original Phase 1 Model 4 used a different epsilon, set cfg['rms_eps'] "
            "to that exact value before constructing the model."
        )

REQUIRED_CFG = ["d_model", "n_layers", "n_heads", "n_kv_heads", "ffn_dim",
                "max_positions", "rope_base", "rms_eps"]
missing_cfg = [k for k in REQUIRED_CFG if k not in cfg]
if missing_cfg:
    raise KeyError(f"Checkpoint config missing: {missing_cfg}")
if int(cfg.get("vocab_size", len(tokenizer))) != len(tokenizer):
    raise ValueError(f"Tokenizer/checkpoint vocab mismatch: {len(tokenizer)} vs {cfg.get('vocab_size')}")
if MAX_SEQ_LEN > int(cfg["max_positions"]):
    raise ValueError(f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds checkpoint max_positions={cfg['max_positions']}")

model = EfficientDecoderLM(cfg, len(tokenizer))
model.load_state_dict(ckpt["model"], strict=True)     # raises on any missing/unexpected key
model.to(DEVICE).eval()
del ckpt; gc.collect()
print(f"Loaded Model 4: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print("Architecture:", {k: cfg[k] for k in REQUIRED_CFG}, "| dropout:", cfg.get("dropout", 0.0))

Loaded Model 4: 99.3M parameters
Architecture: {'d_model': 640, 'n_layers': 17, 'n_heads': 10, 'n_kv_heads': 2, 'ffn_dim': 2048, 'max_positions': 2560, 'rope_base': 10000.0, 'rms_eps': 1e-05} | dropout: 0.0


In [7]:
# 6. Reuse exact LoRA modules and injection logic\n# 6. Attach standard LoRA adapters to all selected Model-4 projections
# Standard LoRA: y = W0(x) + (alpha / r) * B(A(dropout(x))).
# A is Kaiming-initialised and B is zero-initialised, so the initial adapted
# network is exactly function-preserving relative to the frozen Phase-1 model.

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError(f"LoRA expected nn.Linear, got {type(base)}")
        if rank <= 0:
            raise ValueError("LoRA rank must be positive")
        self.base = base
        self.base.requires_grad_(False)
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.lora_dropout = nn.Dropout(float(dropout))
        self.lora_A = nn.Parameter(torch.empty(self.rank, base.in_features,
                                               device=base.weight.device, dtype=base.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, self.rank,
                                               device=base.weight.device, dtype=base.weight.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base(x)
        low_rank = F.linear(F.linear(self.lora_dropout(x), self.lora_A), self.lora_B)
        return base_out + self.scaling * low_rank


def freeze_all(module):
    for p in module.parameters():
        p.requires_grad = False


def inject_lora(model, targets=LORA_TARGETS, rank=LORA_RANK,
                alpha=LORA_ALPHA, dropout=LORA_DROPOUT):
    freeze_all(model)
    replaced = []
    for i, block in enumerate(model.blocks):
        groups = (
            ("attn", block.attn, ("q_proj", "k_proj", "v_proj", "o_proj")),
            ("ff", block.ff, ("gate_proj", "up_proj", "down_proj")),
        )
        for group_name, module, names in groups:
            for name in names:
                if name in targets:
                    old = getattr(module, name)
                    setattr(module, name, LoRALinear(old, rank, alpha, dropout))
                    replaced.append(f"blocks.{i}.{group_name}.{name}")
    return replaced


def is_lora_key(key):
    return key.endswith(".lora_A") or key.endswith(".lora_B")


def lora_state_dict(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()
            if is_lora_key(k)}


def load_lora_state_dict(module, state):
    expected = {k for k in module.state_dict() if is_lora_key(k)}
    if set(state) != expected:
        raise RuntimeError(
            f"LoRA adapter keys mismatch: missing={sorted(expected-set(state))[:4]}, "
            f"unexpected={sorted(set(state)-expected)[:4]}"
        )
    module.load_state_dict(state, strict=False)


# Verify the original model function is unchanged at adapter initialisation.
model.eval()
probe_ids = (torch.arange(48, device=DEVICE) % (len(tokenizer) - 4) + 4)[None, :]
with torch.inference_mode():
    ref_logits = model(probe_ids).float()
replaced = inject_lora(model)
model.eval()
with torch.inference_mode():
    adapted_logits = model(probe_ids).float()
identity_diff = (ref_logits - adapted_logits).abs().max().item()
assert identity_diff < 1e-5, f"LoRA init changed model output: max diff={identity_diff:.3e}"

expected_modules = int(cfg["n_layers"]) * len(LORA_TARGETS)
assert len(replaced) == expected_modules, (len(replaced), expected_modules)
lora_params = [(n, p) for n, p in model.named_parameters()
               if p.requires_grad and (".lora_A" in n or ".lora_B" in n)]
assert len(lora_params) == 2 * expected_modules
assert all(torch.count_nonzero(p.detach()).item() == 0
           for n, p in lora_params if ".lora_B" in n)

trainable = sum(p.numel() for _, p in lora_params)
total = sum(p.numel() for p in model.parameters())
assert all(not p.requires_grad for n, p in model.named_parameters()
           if ".base." in n or n.startswith(("token_emb.", "final_norm.", "blocks.")) and
           ".lora_A" not in n and ".lora_B" not in n)

print("Injected LoRA modules:", len(replaced), "/", expected_modules)
print("Trainable LoRA parameters:", f"{trainable:,}", f"({100*trainable/total:.3f}% of model)")
print("Frozen model parameters:", f"{total-trainable:,}")
print("LoRA scaling alpha/r:", LORA_ALPHA / LORA_RANK)
print("Identity-at-initialisation max-logit difference:", f"{identity_diff:.3e}")
print("Projection shapes:",
      "Q", tuple(model.blocks[0].attn.q_proj.base.weight.shape),
      "K", tuple(model.blocks[0].attn.k_proj.base.weight.shape),
      "V", tuple(model.blocks[0].attn.v_proj.base.weight.shape),
      "O", tuple(model.blocks[0].attn.o_proj.base.weight.shape))


Injected LoRA modules: 119 / 119
Trainable LoRA parameters: 3,307,520 (3.223% of model)
Frozen model parameters: 99,309,440
LoRA scaling alpha/r: 2.0
Identity-at-initialisation max-logit difference: 0.000e+00
Projection shapes: Q (640, 640) K (128, 640) V (128, 640) O (640, 640)


In [8]:
# 6b. Load the trained Model-4 LoRA adapters into the frozen teacher and rename the model.
lora_obj = load_torch(MODEL4_LORA_BEST_PATH)
print("LoRA checkpoint top-level keys:", list(lora_obj.keys()) if isinstance(lora_obj, dict) else type(lora_obj))
_key = next((k for k in ("adapters","adapter","lora","lora_state_dict","lora_state") if isinstance(lora_obj,dict) and k in lora_obj), None)
if _key is None and isinstance(lora_obj, dict) and all(isinstance(k,str) and is_lora_key(k) for k in lora_obj):
    _adapter_state = lora_obj
else:
    assert _key, f"Unknown LoRA checkpoint layout; keys={list(lora_obj)[:15] if isinstance(lora_obj,dict) else type(lora_obj)}"
    _adapter_state = lora_obj[_key]
load_lora_state_dict(model, _adapter_state)

teacher = model; del model; gc.collect()
teacher.to(DEVICE).eval()
for p in teacher.parameters(): p.requires_grad_(False)
print("Teacher = Model 4 + trained LoRA (frozen). Adapter ckpt keys:", {k:(v if not isinstance(v,dict) else "...") for k,v in lora_obj.items() if k!=_key})

LoRA checkpoint top-level keys: ['adapters', 'config', 'lora', 'global_step', 'val_loss', 'base_checkpoint']
Teacher = Model 4 + trained LoRA (frozen). Adapter ckpt keys: {'config': '...', 'lora': '...', 'global_step': 2505, 'val_loss': 1.297139384631602, 'base_checkpoint': '/kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt'}


In [9]:
# 7. Reuse exact Model-2 architecture from your Phase-1 notebook\n# Cell 9 — ~80M Decoder-only causal Transformer with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        # Training / ordinary prefill: use SDPA's efficient causal path.
        # Left-padded batched prefill needs an explicit causal + key-valid mask.
        # Cached one-token decoding already has only past/current keys, so a key-valid mask is enough.
        attn_mask = None
        is_causal = False

        if key_valid is not None:
            key_mask = key_valid[:, None, None, :]  # (B,1,1,S), True = real key
            if past_len == 0 and T > 1:
                causal = torch.ones((T, T), dtype=torch.bool, device=x.device).tril()
                attn_mask = key_mask & causal[None, None, :, :]

                # Left-padding creates early PAD query rows with no valid causal key.
                # Giving each such row access to itself avoids fully-masked SDPA rows
                # on kernels/versions that may otherwise produce NaNs. Real-token
                # attention is unchanged because real positions already include self.
                eye = torch.eye(T, dtype=torch.bool, device=x.device)[None, None, :, :]
                attn_mask = attn_mask | eye
            else:
                attn_mask = key_mask
        elif past_kv is None:
            is_causal = True

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)

        new_kv = (k, v) if use_cache else None
        return y, new_kv


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.ln1(x),
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x, new_kv


class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-style scaled residual projections.
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_ids=None,
        key_valid=None,
    ):
        B, T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_cache = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x, new_cache

    def project(self, hidden):
        # Tied token/output embedding.
        return F.linear(hidden, self.token_emb.weight)

In [10]:
# 7b. Build Model-2 student from its own checkpoint config and strictly load Phase-1 weights.
student_obj = load_torch(MODEL2_BASE_PATH)
cfg2 = dict(student_obj.get("config", {}))
print("Model-2 config:", cfg2)
def _cg(*names, default=None):
    for n in names:
        if n in cfg2: return cfg2[n]
    if default is None: raise KeyError(f"Model-2 config missing one of {names}; set values from the printed config")
    return default
student = DecoderOnlyLM(vocab_size=len(tokenizer), d_model=int(_cg("d_model")), n_layers=int(_cg("n_layers","num_layers")),
                        n_heads=int(_cg("n_heads","num_heads")), ffn_dim=int(_cg("ffn_dim","d_ff")),
                        dropout=float(_cg("dropout",default=0.0)), max_positions=int(_cg("max_positions","max_len","max_seq_len"))).to(DEVICE)
_sd = student_obj.get("model", student_obj.get("state_dict", student_obj))
student.load_state_dict(_sd, strict=True); del student_obj, _sd; gc.collect()
STUDENT_DROPOUT_TRAIN = 0.1   # light regularisation while distilling (restored to eval at inference)
for m_ in student.modules():
    if isinstance(m_, nn.Dropout): m_.p = STUDENT_DROPOUT_TRAIN
for b_ in student.blocks: b_.attn.dropout = STUDENT_DROPOUT_TRAIN
# Context must fit BOTH models.
MAX_SEQ_LEN = min(MAX_SEQ_LEN, student.max_positions, teacher.max_positions)
assert MAX_SEQ_LEN >= 1200, f"student max_positions too small: {student.max_positions}"
print(f"Student params {sum(p.numel() for p in student.parameters())/1e6:.1f}M | MAX_SEQ_LEN={MAX_SEQ_LEN}")
# Both models must produce logits over the same vocabulary for the same ids.
with torch.no_grad():
    _p = torch.arange(32, device=DEVICE)[None,:] % (len(tokenizer)-4) + 4
    _t = F.linear(teacher.forward_hidden(_p), teacher.token_emb.weight)
    _s = student.project(student.forward_hidden(_p)[0])
    assert _t.shape == _s.shape and torch.isfinite(_t).all() and torch.isfinite(_s).all()
print("Interface gate passed: teacher/student logits", tuple(_t.shape)); del _t,_s,_p

Model-2 config: {'architecture': 'decoder_only_causal', 'vocab_size': 24576, 'd_model': 640, 'n_layers': 13, 'n_heads': 10, 'ffn_dim': 2560, 'dropout': 0.0, 'max_positions': 2048, 'seq_len': 1536, 'prompt_p10_tokens': 1050, 'prompt_p95_tokens': 1139, 'expected_cont_tokens': 384, 'prompt_split_min': 1050, 'prompt_split_max': 1139}
Student params 79.7M | MAX_SEQ_LEN=2048
Interface gate passed: teacher/student logits (1, 32, 24576)


## 9. Load both tokenizers and prove ID-level compatibility

Equal vocabulary size is not enough. We compare every vocabulary entry in order and special-token IDs. If the files differ, the notebook stops before training.

In [11]:
tok_teacher = LegalBPE.load_json(MODEL4_TOKENIZER_PATH)
tok_student = LegalBPE.load_json(MODEL2_TOKENIZER_PATH)
assert len(tok_teacher) == len(tok_student), (len(tok_teacher),len(tok_student))
# LegalBPE uses ordered itos/stoi tables; compare complete ordered vocabularies.
assert tok_teacher.itos == tok_student.itos, "Tokenizer token-ID mappings differ: direct logit KD is invalid."
for attr in ["pad_token_id","unk_token_id","bos_token_id","eos_token_id"]:
    assert getattr(tok_teacher,attr) == getattr(tok_student,attr), f"Special token mismatch: {attr}"
tokenizer = tok_teacher
VOCAB_SIZE = len(tokenizer)
print("Tokenizer mappings identical:", VOCAB_SIZE, "tokens")


Tokenizer mappings identical: 24576 tokens


## 12. Dataset, duplicate-grouped split, and tokenization

The next cell intentionally uses the same Phase-6 prompt layout as your LoRA teacher experiment. We use only official training data; no external corpus or model.

In [12]:
# 7. Load Phase 6 data and build a duplicate-grouped train/validation split
def load_records(path):
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):                      # id-keyed JSON
        return [{"id": str(k), **v} for k, v in obj.items() if isinstance(v, dict)]
    if not isinstance(obj, list):
        raise TypeError("Expected a JSON list or an ID-keyed object")
    return obj

_WS_RE = re.compile(r"\s+")
def doc_group_key(text):
    # Normalised-document hash: exact duplicate documents always land in the same split.
    return hashlib.sha1(_WS_RE.sub(" ", text).strip().lower().encode("utf-8")).hexdigest()

train_records = load_records(TRAIN_JSON_PATH)
test_records = load_records(TEST_JSON_PATH)

clean = []
for r in train_records:
    rid, doc, summ = str(r.get("id", "")), r.get("document", ""), r.get("summary", "")
    if rid and isinstance(doc, str) and doc.strip() and isinstance(summ, str) and summ.strip():
        clean.append({"id": rid, "document": doc.strip(), "summary": summ.strip()})
for r in clean:
    r["group"] = doc_group_key(r["document"])
print(f"Raw train: {len(train_records)} | usable: {len(clean)} | dropped: {len(train_records) - len(clean)} | test: {len(test_records)}")
assert len({r["id"] for r in clean}) == len(clean), "duplicate ids in train.json"

# Validate the test file up-front so inference cannot fail halfway through.
for i, r in enumerate(test_records):
    if not str(r.get("id", "")) or not isinstance(r.get("document"), str):
        raise ValueError(f"Invalid test record at index {i}")
assert len({str(r["id"]) for r in test_records}) == len(test_records), "duplicate ids in test.json"

groups = defaultdict(list)
for r in clean:
    groups[r["group"]].append(r)
group_keys = list(groups)
random.Random(SEED).shuffle(group_keys)
target_val = max(1, int(len(clean) * VAL_FRACTION))
val_keys, n_val = set(), 0
for key in group_keys:
    if n_val >= target_val:
        break
    val_keys.add(key); n_val += len(groups[key])

train_rows = [r for r in clean if r["group"] not in val_keys]
val_rows = [r for r in clean if r["group"] in val_keys]
random.Random(SEED + 1).shuffle(val_rows)     # fixed random order -> "first N" is a random sample
assert train_rows and val_rows
assert {r["id"] for r in train_rows}.isdisjoint(r["id"] for r in val_rows)
assert {r["group"] for r in train_rows}.isdisjoint(r["group"] for r in val_rows)

train_groups = {r["group"] for r in train_rows}
test_overlap = sum(doc_group_key(r["document"].strip()) in train_groups for r in test_records)
print(f"Train={len(train_rows)} | validation={len(val_rows)} | unique documents={len(groups)} "
      f"| duplicated-doc rows={len(clean) - len(groups)}")
print(f"Test documents that also appear (exactly) in train: {test_overlap}")

Raw train: 7030 | usable: 7028 | dropped: 2 | test: 100
Train=6677 | validation=351 | unique documents=7001 | duplicated-doc rows=27
Test documents that also appear (exactly) in train: 0


In [13]:
# 8a. Source selection (head+tail, as in the base prompt layout). Returns (ids, was_truncated).
def select_source_ids(document):
    ids = tokenizer.encode(document)
    return ids, len(ids) > MAX_SOURCE_TOKENS

In [14]:
# KD hyper-parameters (v2). Teacher logits are computed ONLINE (teacher-forced, no_grad): a full
# [tokens x 24576-vocab] fp16 cache would need tens of GB, so caching is unnecessary and unsafe.
EPOCHS = 3
LEARNING_RATE = 1e-4          # full fine-tune of an 80M student
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
KD_TEMPERATURE = 2.0
HARD_LOSS_WEIGHT = 0.5
KD_LOSS_WEIGHT = 0.5
MAX_GRAD_NORM = 1.0
GRAD_ACCUM_STEPS = 8
VAL_EVERY_STEPS = 150
TRAIN_TIME_BUDGET_MIN = 150   # safety: stop training (keeping best ckpt) if exceeded

In [15]:
# 8. Build causal-LM examples:  <s> + instruction + source window + "SUMMARY:" separator + summary + </s>
# Only the summary (and EOS) is supervised. The prompt/separator are never truncated: the *source*
# window shrinks first when prompt + source + target would exceed MAX_SEQ_LEN.
PROMPT_TEMPLATE = "Summarize the following Indian court judgment accurately. Preserve the key facts, legal issues, reasoning, decision, and important citations. Do not invent facts.\n\nJUDGMENT:\n"
PROMPT_IDS = tokenizer.encode(PROMPT_TEMPLATE)
SEP_IDS = tokenizer.encode("\n\nSUMMARY:\n")
PREFIX_OVERHEAD = 1 + len(PROMPT_IDS) + len(SEP_IDS)       # BOS + instruction + separator
min_source_budget = MAX_SEQ_LEN - PREFIX_OVERHEAD - MAX_TARGET_TOKENS - 1
if min_source_budget < 256:
    raise ValueError(f"MAX_SEQ_LEN too small: only {min_source_budget} source tokens left for the longest target.")

def head_tail(ids, budget):
    # Keep the opening 2/3 and the closing 1/3 of the budget (facts early, decision late).
    # Nested property: head_tail(head_tail(x, B1), B2) == head_tail(x, B2) whenever B2 <= B1.
    assert budget >= 1
    if len(ids) <= budget:
        return ids
    left = budget * 2 // 3
    right = budget - left
    return ids[:left] + ids[-right:]

def build_prompt_ids(src_ids, budget):
    return [BOS_ID] + PROMPT_IDS + head_tail(src_ids, budget) + SEP_IDS

def tokenize_rows(rows, name):
    examples, n_trunc, t0 = [], 0, time.time()
    for i, r in enumerate(rows):
        src, was_cut = select_source_ids(r["document"])
        n_trunc += int(was_cut)
        tgt = tokenizer.encode(r["summary"])[:MAX_TARGET_TOKENS]
        budget = min(MAX_SOURCE_TOKENS, MAX_SEQ_LEN - PREFIX_OVERHEAD - len(tgt) - 1)
        prefix = build_prompt_ids(src, budget)
        ids = prefix + tgt + [EOS_ID]
        assert len(ids) <= MAX_SEQ_LEN
        examples.append((np.asarray(ids, dtype=np.int32), len(prefix)))
        if (i + 1) % 500 == 0:
            print(f"  [{name}] tokenized {i + 1}/{len(rows)} ({time.time() - t0:.0f}s)", flush=True)
    print(f"{name}: {len(examples)} examples | source truncated: {n_trunc} ({100 * n_trunc / max(1, len(rows)):.1f}%)")
    return examples

print("Tokenizing (pure-Python BPE; this can take a few minutes) ...")
train_examples = tokenize_rows(train_rows, "train")
val_examples = tokenize_rows(val_rows, "val")

class SummarizationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        ids, prefix_len = self.examples[i]
        ids_t = torch.from_numpy(ids.astype(np.int64))
        labels = ids_t.clone()
        labels[:prefix_len] = -100
        return {"input_ids": ids_t, "labels": labels}

def collate(batch):
    L = max(x["input_ids"].numel() for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, x in enumerate(batch):
        n = x["input_ids"].numel()
        ids[i, :n] = x["input_ids"]; labels[i, :n] = x["labels"]
    return {"input_ids": ids, "labels": labels}

train_ds = SummarizationDataset(train_examples)
val_ds = SummarizationDataset(val_examples)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate, num_workers=0)

lens = np.array([len(e[0]) for e in train_examples])
print(f"Train sequence length: mean={lens.mean():.0f} p95={np.percentile(lens, 95):.0f} max={lens.max()}")

Tokenizing (pure-Python BPE; this can take a few minutes) ...
  [train] tokenized 500/6677 (4s)
  [train] tokenized 1000/6677 (8s)
  [train] tokenized 1500/6677 (12s)
  [train] tokenized 2000/6677 (15s)
  [train] tokenized 2500/6677 (18s)
  [train] tokenized 3000/6677 (22s)
  [train] tokenized 3500/6677 (26s)
  [train] tokenized 4000/6677 (30s)
  [train] tokenized 4500/6677 (34s)
  [train] tokenized 5000/6677 (37s)
  [train] tokenized 5500/6677 (41s)
  [train] tokenized 6000/6677 (44s)
  [train] tokenized 6500/6677 (48s)
train: 6677 examples | source truncated: 6552 (98.1%)
val: 351 examples | source truncated: 343 (97.7%)
Train sequence length: mean=2035 p95=2048 max=2048


## KD loss (summary positions only)

In [16]:
def summarization_kd_loss(student_logits, teacher_logits, labels, temperature=2.0,
                           hard_weight=0.5, kd_weight=0.5):
    # logits [B,T,V], labels [B,T], ignored prompt positions = -100
    valid = labels.ne(-100)
    if not valid.any():
        raise ValueError("Batch has no supervised summary tokens.")
    s = student_logits[valid].float()
    t = teacher_logits[valid].float()
    y = labels[valid]
    hard = F.cross_entropy(s, y)
    log_ps = F.log_softmax(s / temperature, dim=-1)
    pt = F.softmax(t / temperature, dim=-1)
    soft = F.kl_div(log_ps, pt, reduction="batchmean") * (temperature ** 2)
    return hard_weight * hard + kd_weight * soft, hard.detach(), soft.detach()


In [17]:
from datetime import datetime
from zoneinfo import ZoneInfo
def ist_now():
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%d %I:%M:%S %p IST")
def progress_line(done, total, started, label="work"):
    elapsed=time.time()-started
    rate=done/max(elapsed,1e-9)
    eta=(total-done)/max(rate,1e-9)
    print(f"[{ist_now()}] {label}: {done}/{total} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min", flush=True)
print("Notebook initialized at", ist_now())


Notebook initialized at 2026-11-02 01:44:56 PM IST


In [18]:
# 15b. KD training: student full fine-tune; teacher frozen online. Loss on summary tokens only.
def kd_batch_loss(batch, train=True):
    ids = batch["input_ids"].to(DEVICE); labels = batch["labels"].to(DEVICE)
    tgt = labels[:,1:]; valid = tgt.ne(-100)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda")):
        th = teacher.forward_hidden(ids)[:, :-1][valid]
        t_logits = F.linear(th, teacher.token_emb.weight)
    with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda")):
        sh = student.forward_hidden(ids)[0][:, :-1][valid]
        s_logits = student.project(sh)
    y = tgt[valid]
    s = s_logits.float(); t = t_logits.float()
    hard = F.cross_entropy(s, y)
    soft = F.kl_div(F.log_softmax(s/KD_TEMPERATURE,-1), F.softmax(t/KD_TEMPERATURE,-1), reduction="batchmean")*KD_TEMPERATURE**2
    return HARD_LOSS_WEIGHT*hard + KD_LOSS_WEIGHT*soft, float(hard), float(soft)

@torch.no_grad()
def eval_student_ce(max_docs=60):
    student.eval(); tot=0.0; n=0
    for i,b in enumerate(val_loader):
        if i>=max_docs: break
        ids=b["input_ids"].to(DEVICE); lab=b["labels"].to(DEVICE)
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda")):
            h=student.forward_hidden(ids)[0][:, :-1]; tg=lab[:,1:]; v=tg.ne(-100)
            lg=student.project(h[v])
        tot+=float(F.cross_entropy(lg.float(), tg[v])); n+=1
    student.train(); return tot/max(1,n)

# Preflight: verify teacher/student distributions are finite and vocabulary-aligned
_probe = next(iter(DataLoader(train_ds, batch_size=1, shuffle=False, collate_fn=collate)))
with torch.no_grad():
    _ids = _probe["input_ids"].to(DEVICE); _lab = _probe["labels"].to(DEVICE)
    _valid = _lab[:,1:].ne(-100)
    _th = teacher.forward_hidden(_ids)[:,:-1][_valid]
    _sh = student.forward_hidden(_ids)[0][:,:-1][_valid]
    _tl = F.linear(_th, teacher.token_emb.weight).float()
    _sl = student.project(_sh).float()
    assert _tl.shape == _sl.shape and _tl.shape[-1] == VOCAB_SIZE
    assert torch.isfinite(_tl).all() and torch.isfinite(_sl).all()
    _kd_probe = F.kl_div(F.log_softmax(_sl/KD_TEMPERATURE,-1),
                         F.softmax(_tl/KD_TEMPERATURE,-1), reduction="batchmean") * KD_TEMPERATURE**2
    assert torch.isfinite(_kd_probe), "KD preflight produced non-finite KL"
print("KD preflight passed:", tuple(_tl.shape), "| KL", float(_kd_probe))
del _probe,_ids,_lab,_valid,_th,_sh,_tl,_sl,_kd_probe
gc.collect(); torch.cuda.empty_cache() if DEVICE.type=="cuda" else None

train_loader = DataLoader(train_ds, batch_size=MICRO_BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=0,
                          generator=torch.Generator().manual_seed(SEED))
decay=[p for n,p in student.named_parameters() if p.ndim>=2 and "token_emb" not in n]
nodecay=[p for n,p in student.named_parameters() if not (p.ndim>=2 and "token_emb" not in n)]
optimizer=torch.optim.AdamW([{"params":decay,"weight_decay":WEIGHT_DECAY},{"params":nodecay,"weight_decay":0.0}],
                            lr=LEARNING_RATE, betas=(0.9,0.95))
steps_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS); total_steps=steps_per_epoch*EPOCHS
warm=max(1,int(total_steps*WARMUP_RATIO))
sched=torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s:(s+1)/warm if s<warm else 0.1+0.9*0.5*(1+math.cos(math.pi*min(1.0,(s-warm)/max(1,total_steps-warm)))))
scaler=torch.amp.GradScaler("cuda", enabled=(DEVICE.type=="cuda"))
best_ce=eval_student_ce(); print(f"Baseline student val CE: {best_ce:.4f} | total optimizer steps: {total_steps}")
history=[{"step":0,"val_ce":best_ce}]; gstep=0; t_start=time.time(); stop=False
student.train()
for ep in range(EPOCHS):
    run=[0,0,0,0]
    for bi,batch in enumerate(train_loader):
        loss,hd,sf=kd_batch_loss(batch)
        if not torch.isfinite(loss): print("non-finite loss, skipping batch"); optimizer.zero_grad(set_to_none=True); continue
        scaler.scale(loss/GRAD_ACCUM_STEPS).backward(); run[0]+=float(loss);run[1]+=hd;run[2]+=sf;run[3]+=1
        if (bi+1)%GRAD_ACCUM_STEPS==0 or bi+1==len(train_loader):
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(student.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); sched.step(); gstep+=1
            if gstep%10==0:
                n_=max(1,run[3]); print(f"ep {ep+1}/{EPOCHS} step {gstep}/{total_steps} loss {run[0]/n_:.3f} hard {run[1]/n_:.3f} kd {run[2]/n_:.3f}",flush=True)
                progress_line(gstep,total_steps,t_start,"KD training"); run=[0,0,0,0]
            if gstep%VAL_EVERY_STEPS==0 or gstep==total_steps:
                ce=eval_student_ce(); history.append({"step":gstep,"val_ce":ce}); print(f"  val CE @ {gstep}: {ce:.4f}",flush=True)
                if ce<best_ce:
                    best_ce=ce; torch.save({"model":student.state_dict(),"step":gstep,"val_ce":ce},ROOT/"student_best.pt")
            if (time.time()-t_start)/60>TRAIN_TIME_BUDGET_MIN: print("Time budget reached; stopping."); stop=True; break
    if stop: break
if not (ROOT/"student_best.pt").exists(): torch.save({"model":student.state_dict(),"step":gstep,"val_ce":best_ce},ROOT/"student_best.pt")
json.dump(history,open(ROOT/"training_history.json","w"),indent=2)
bst=torch.load(ROOT/"student_best.pt",map_location="cpu",weights_only=False); student.load_state_dict(bst["model"]); student.eval()
del teacher, optimizer; gc.collect(); torch.cuda.empty_cache()
print("Restored best student: step",bst["step"],"val CE",bst["val_ce"])

KD preflight passed: (801, 24576) | KL 28.929096221923828
Baseline student val CE: 20.0740 | total optimizer steps: 2505


/tmp/ipykernel_48/50373710.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return HARD_LOSS_WEIGHT*hard + KD_LOSS_WEIGHT*soft, float(hard), float(soft)
/tmp/ipykernel_48/50373710.py:68: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); sched.step(); gstep+=1


ep 1/3 step 10/2505 loss 26.526 hard 25.465 kd 27.587
[2026-11-02 01:45:17 PM IST] KD training: 10/2505 | elapsed 0.2 min | ETA 61.9 min
ep 1/3 step 20/2505 loss 18.949 hard 18.425 kd 19.472
[2026-11-02 01:45:32 PM IST] KD training: 20/2505 | elapsed 0.5 min | ETA 61.8 min
ep 1/3 step 30/2505 loss 7.529 hard 7.466 kd 7.593
[2026-11-02 01:45:47 PM IST] KD training: 30/2505 | elapsed 0.8 min | ETA 62.1 min
ep 1/3 step 40/2505 loss 4.964 hard 5.526 kd 4.402
[2026-11-02 01:46:03 PM IST] KD training: 40/2505 | elapsed 1.0 min | ETA 62.4 min
ep 1/3 step 50/2505 loss 4.500 hard 5.131 kd 3.870
[2026-11-02 01:46:19 PM IST] KD training: 50/2505 | elapsed 1.3 min | ETA 62.8 min
ep 1/3 step 60/2505 loss 4.290 hard 4.935 kd 3.646
[2026-11-02 01:46:36 PM IST] KD training: 60/2505 | elapsed 1.6 min | ETA 63.4 min
ep 1/3 step 70/2505 loss 4.160 hard 4.800 kd 3.519
[2026-11-02 01:46:52 PM IST] KD training: 70/2505 | elapsed 1.8 min | ETA 63.5 min
ep 1/3 step 80/2505 loss 3.951 hard 4.601 kd 3.302
[2026

## Decoding, validation and submission

In [19]:
# 17a. Batched KV-cached generation for the student (left padding + key_valid + explicit position ids).
BAD_IDS=[PAD_ID,tokenizer.unk_token_id,BOS_ID]
def make_prompt(document):
    src,_=select_source_ids(document)
    budget=min(MAX_SOURCE_TOKENS, MAX_SEQ_LEN-PREFIX_OVERHEAD-GEN_MAX_NEW_TOKENS-1)
    return build_prompt_ids(src,budget)

@torch.no_grad()
def _gen_chunk(prompts, min_new, ngram, rep, max_new=GEN_MAX_NEW_TOKENS):
    B=len(prompts); L=max(len(p) for p in prompts)
    ids=torch.full((B,L),PAD_ID,dtype=torch.long,device=DEVICE); valid=torch.zeros((B,L),dtype=torch.bool,device=DEVICE)
    for i,p in enumerate(prompts):
        ids[i,L-len(p):]=torch.tensor(p,device=DEVICE); valid[i,L-len(p):]=True
    pos=(valid.long().cumsum(1)-1).clamp(min=0)
    with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
        h,cache=student.forward_hidden(ids,use_cache=True,position_ids=pos,key_valid=valid)
        logits=student.project(h[:,-1])
    nxt_pos=pos[:,-1]+1
    gen=[[] for _ in range(B)]; done=[False]*B; seen=[set() for _ in range(B)]; ban=[{} for _ in range(B)]
    for step in range(max_new):
        z=logits.float().clone(); z[:,BAD_IDS]=-float("inf")
        if step<min_new: z[:,EOS_ID]=-float("inf")
        for i in range(B):
            if done[i]: continue
            if rep!=1.0 and seen[i]:
                u=torch.tensor(list(seen[i]),device=DEVICE); v=z[i,u]; z[i,u]=torch.where(v<0,v*rep,v/rep)
            if ngram>=2 and len(gen[i])>=ngram-1:
                b=ban[i].get(tuple(gen[i][-(ngram-1):]))
                if b: z[i,torch.tensor(list(b),device=DEVICE)]=-float("inf")
        nxt=z.argmax(-1).tolist()
        for i in range(B):
            if done[i]: continue
            t=nxt[i]
            if t==EOS_ID: done[i]=True; continue
            if ngram>=2 and len(gen[i])>=ngram-1: ban[i].setdefault(tuple(gen[i][-(ngram-1):]),set()).add(t)
            gen[i].append(t); seen[i].add(t)
        if all(done): break
        valid=torch.cat([valid,torch.ones((B,1),dtype=torch.bool,device=DEVICE)],1)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
            h,cache=student.forward_hidden(torch.tensor(nxt,device=DEVICE)[:,None],past_key_values=cache,use_cache=True,
                                           position_ids=nxt_pos[:,None],key_valid=valid)
            logits=student.project(h[:,-1])
        nxt_pos=nxt_pos+1
    return [tokenizer.decode(g_).strip() for g_ in gen]

FALLBACK_SUMMARY="The judgment could not be summarized from the supplied context."
def generate_summaries(documents, gc_cfg, bs=None, desc="gen"):
    bs=bs or GEN_BATCH_SIZE
    prompts=[make_prompt(d) for d in documents]; order=sorted(range(len(prompts)),key=lambda i:len(prompts[i]))
    out=[None]*len(prompts); i=0; t0=time.time()
    min_new=int(gc_cfg["min_frac"]*GEN_MAX_NEW_TOKENS)
    while i<len(order):
        idx=order[i:i+bs]
        try:
            res=_gen_chunk([prompts[j] for j in idx],min_new,gc_cfg["ngram"],gc_cfg["rep"])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs==1: raise
            bs=max(1,bs//2); print("OOM -> batch size",bs); continue
        for j,r in zip(idx,res): out[j]=r if r.strip() else FALLBACK_SUMMARY
        i+=len(idx)
        if (i//max(1,len(idx)))%5==0: progress_line(i,len(order),t0,desc)
    return out

In [20]:
# 17b. Metrics + decoding-grid tuning on a slice disjoint from the scored validation docs.
import importlib, subprocess, sys
for _m,_p in (("rouge_score","rouge-score"),("bert_score","bert-score")):
    try: importlib.import_module(_m)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",_p])
from rouge_score import rouge_scorer
from bert_score import BERTScorer
_rs=rouge_scorer.RougeScorer(["rouge1","rougeL"],use_stemmer=True)
_bs=BERTScorer(lang="en",batch_size=BERTSCORE_BATCH_SIZE,device=str(DEVICE),rescale_with_baseline=False)
def score_all(preds,refs):
    _,_,f=_bs.score([p if p.strip() else "." for p in preds],refs)
    sc=[_rs.score(r,p) for p,r in zip(preds,refs)]
    return {"rouge1_f1":float(np.mean([s["rouge1"].fmeasure for s in sc])),"rougeL_f1":float(np.mean([s["rougeL"].fmeasure for s in sc])),
            "bertscore_f1":float(f.mean()),"mean_words":float(np.mean([len(p.split()) for p in preds]))}
score_rows=val_rows[:VAL_SCORE_EXAMPLES]; tune_rows=val_rows[VAL_SCORE_EXAMPLES:VAL_SCORE_EXAMPLES+N_TUNE_DOCS]
if len(tune_rows)<8: tune_rows=val_rows[-min(N_TUNE_DOCS,len(val_rows)):]   # tiny val set fallback
grid_results=[]
for cfg_ in GEN_GRID:
    t0=time.time(); preds=generate_summaries([r["document"] for r in tune_rows],cfg_,desc=cfg_["name"])
    m=score_all(preds,[r["summary"] for r in tune_rows]); m.update(cfg_); m["composite"]=composite_score(m); m["seconds"]=time.time()-t0
    grid_results.append(m); print({k:(round(v,4) if isinstance(v,float) else v) for k,v in m.items()},flush=True)
BEST_GEN=max(grid_results,key=lambda x:x["composite"])
json.dump({"best":BEST_GEN,"all":grid_results},open(ROOT/"decode_sweep.json","w"),indent=2)
print("Selected decoding (primary metric: BERTScore F1):",BEST_GEN["name"])

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'rouge1_f1': 0.2293, 'rougeL_f1': 0.2102, 'bertscore_f1': 0.7489, 'mean_words': 400.0, 'name': 'greedy', 'min_frac': 0.0, 'ngram': 0, 'rep': 1.0, 'composite': 0.7489, 'seconds': 56.1238}
{'rouge1_f1': 0.2293, 'rougeL_f1': 0.2102, 'bertscore_f1': 0.7489, 'mean_words': 400.0, 'name': 'greedy+min', 'min_frac': 0.45, 'ngram': 0, 'rep': 1.0, 'composite': 0.7489, 'seconds': 54.8313}
{'rouge1_f1': 0.2658, 'rougeL_f1': 0.2168, 'bertscore_f1': 0.7484, 'mean_words': 400.0, 'name': 'ng12+min', 'min_frac': 0.45, 'ngram': 12, 'rep': 1.0, 'composite': 0.7484, 'seconds': 55.685}
{'rouge1_f1': 0.2563, 'rougeL_f1': 0.2141, 'bertscore_f1': 0.7514, 'mean_words': 400.0, 'name': 'ng20+min', 'min_frac': 0.45, 'ngram': 20, 'rep': 1.0, 'composite': 0.7514, 'seconds': 55.1857}
{'rouge1_f1': 0.2626, 'rougeL_f1': 0.2143, 'bertscore_f1': 0.7503, 'mean_words': 400.0, 'name': 'ng20+rep1.05+min', 'min_frac': 0.45, 'ngram': 20, 'rep': 1.05, 'composite': 0.7503, 'seconds': 57.0589}
{'rouge1_f1': 0.2695, 'rougeL_f1': 

In [21]:
# 17c. Final validation (scored once) + resumable test predictions + submission zip.
val_preds=generate_summaries([r["document"] for r in score_rows],BEST_GEN,desc="validation")
val_metrics=score_all(val_preds,[r["summary"] for r in score_rows]); print("FINAL VALIDATION:",json.dumps(val_metrics,indent=2))
json.dump([{"id":r["id"],"summary":p} for r,p in zip(score_rows,val_preds)],open(ROOT/"validation_predictions.json","w"),ensure_ascii=False,indent=2)

partial=ROOT/"test_predictions_partial.jsonl"; done={}
if RESUME_TEST_PREDICTIONS and partial.exists():
    for line in open(partial,encoding="utf-8"):
        if line.strip(): d=json.loads(line); done[d["id"]]=d["summary"]
    print("Resuming with",len(done),"cached test predictions")
todo=[r for r in test_records if str(r["id"]) not in done]
for s in range(0,len(todo),32):
    chunk=todo[s:s+32]; outs=generate_summaries([r["document"] for r in chunk],BEST_GEN,desc="test")
    with open(partial,"a",encoding="utf-8") as f:
        for r,p in zip(chunk,outs): done[str(r["id"])]=p; f.write(json.dumps({"id":str(r["id"]),"summary":p},ensure_ascii=False)+"\n")
test_preds=[{"id":str(r["id"]),"summary":done[str(r["id"])]} for r in test_records]
assert len(test_preds)==len(test_records) and len({x["id"] for x in test_preds})==len(test_records) and all(x["summary"].strip() for x in test_preds)
json.dump(test_preds,open(ROOT/"prediction.json","w",encoding="utf-8"),ensure_ascii=False,indent=2)
with zipfile.ZipFile(ROOT/"submission.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(ROOT/"prediction.json",arcname="prediction.json")
lens=np.array([len(x["summary"].split()) for x in test_preds])
print(f"Test words mean={lens.mean():.0f} min={lens.min()} max={lens.max()} | fallbacks={sum(x['summary']==FALLBACK_SUMMARY for x in test_preds)}")
json.dump({"experiment":"Phase6-Exp7-Model4LoRA-to-Model2-KD","epochs":EPOCHS,"lr":LEARNING_RATE,"temperature":KD_TEMPERATURE,
           "hard_w":HARD_LOSS_WEIGHT,"kd_w":KD_LOSS_WEIGHT,"best_val_ce":best_ce,"decoding":BEST_GEN,"validation":val_metrics,"seed":SEED},
          open(ROOT/"experiment_metadata.json","w"),indent=2)
print("Submission:",ROOT/"submission.zip")

[2026-11-02 03:00:07 PM IST] validation: 40/100 | elapsed 1.0 min | ETA 1.5 min
[2026-11-02 03:01:08 PM IST] validation: 80/100 | elapsed 2.0 min | ETA 0.5 min
[2026-11-02 03:01:41 PM IST] validation: 100/100 | elapsed 2.6 min | ETA 0.0 min
FINAL VALIDATION: {
  "rouge1_f1": 0.25129076639806414,
  "rougeL_f1": 0.20574357266348733,
  "bertscore_f1": 0.7524781823158264,
  "mean_words": 400.0
}
Test words mean=400 min=400 max=400 | fallbacks=0
Submission: /kaggle/working/phase6_exp7_model4_lora_to_model2_kd/submission.zip
